# ⚗️ OpenMM: Zero to Hero
## A Comprehensive Guide to Molecular Dynamics Simulation in Python

---

**Welcome!** This notebook is your complete guide to OpenMM — from first principles to expert-level molecular dynamics techniques. By the end, you'll be able to:
- Set up and run MD simulations from scratch
- Work with force fields and system preparation
- Perform energy minimization, equilibration and production runs
- Analyze trajectories: RMSD, RMSF, distances, angles
- Run enhanced sampling: metadynamics, replica exchange
- Build custom forces and integrators
- Perform free energy calculations
- Use GPU acceleration

**Prerequisites:** Basic Python, some chemistry/physics background helpful

**Installation:**
```bash
conda install -c conda-forge openmm
conda install -c conda-forge mdtraj parmed nglview  # analysis tools
pip install openmmforcefields  # extra force fields
```

---

## 📚 Table of Contents

1. [Chapter 1: Core Concepts & Architecture](#chapter1)
2. [Chapter 2: Your First Simulation](#chapter2)
3. [Chapter 3: Force Fields](#chapter3)
4. [Chapter 4: System Setup from PDB](#chapter4)
5. [Chapter 5: Integrators & Thermostats](#chapter5)
6. [Chapter 6: Energy Minimization & Equilibration](#chapter6)
7. [Chapter 7: Production Runs & Reporters](#chapter7)
8. [Chapter 8: Solvation & Periodic Boundary Conditions](#chapter8)
9. [Chapter 9: Trajectory Analysis](#chapter9)
10. [Chapter 10: Constraints, Restraints & Custom Forces](#chapter10)
11. [Chapter 11: Barostats & NPT Ensemble](#chapter11)
12. [Chapter 12: Enhanced Sampling Methods](#chapter12)
13. [Chapter 13: Free Energy Calculations](#chapter13)
14. [Chapter 14: GPU Acceleration & Performance](#chapter14)
15. [Chapter 15: Advanced Topics & Real Workflows](#chapter15)

---
<a id='chapter1'></a>
# Chapter 1: Core Concepts & Architecture 🏗️

## What is Molecular Dynamics?

MD simulates the physical movements of atoms over time by integrating Newton's equations of motion:

$$F = ma \quad \Rightarrow \quad a_i = \frac{F_i}{m_i}$$

Forces come from the **force field** — a mathematical description of how atoms interact.

## OpenMM Architecture

```
Topology  ←── defines atoms, bonds, residues, chains
    +
System    ←── particles + forces (built from Topology + ForceField)
    +
Integrator ←── algorithm to advance time (Verlet, Langevin, etc.)
    +
Platform  ←── hardware (CPU, CUDA, OpenCL, Reference)
    ↓
Context   ←── the running simulation state
    ↓
Simulation ←── high-level wrapper (adds Reporters for output)
```

In [ ]:
# Verify your OpenMM installation
import openmm
import openmm.app as app
import openmm.unit as unit

print(f"OpenMM version: {openmm.__version__}")

# Check available platforms
print("\nAvailable platforms:")
for i in range(openmm.Platform.getNumPlatforms()):
    platform = openmm.Platform.getPlatform(i)
    print(f"  {i}: {platform.getName()} (speed={platform.getSpeed()})")

# Standard imports we'll use throughout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, time

print("\nAll imports successful! ✅")

In [ ]:
# ============================================================
# Understanding OpenMM Units
# ============================================================
# OpenMM uses a unit system: nm, ps, kJ/mol, K, bar
# ALWAYS attach units to quantities — it prevents bugs!

# Unit basics
distance  = 1.0 * unit.nanometer
energy    = 1.0 * unit.kilojoule_per_mole
temp      = 300 * unit.kelvin
timestep  = 2.0 * unit.femtoseconds
pressure  = 1.0 * unit.bar

print("Common units in OpenMM:")
print(f"  Distance:    {distance}")
print(f"  Energy:      {energy}")
print(f"  Temperature: {temp}")
print(f"  Timestep:    {timestep}")
print(f"  Pressure:    {pressure}")

# Unit conversion
d_angstrom = distance.in_units_of(unit.angstrom)
d_nm = (3.5 * unit.angstrom).in_units_of(unit.nanometer)
print(f"\n1 nm = {d_angstrom}")
print(f"3.5 Å = {d_nm}")

# Energy conversion
e_kcal = energy.in_units_of(unit.kilocalorie_per_mole)
print(f"1 kJ/mol = {e_kcal}")

# Extract numerical value
T_value = temp.value_in_unit(unit.kelvin)
print(f"\nNumerical value of {temp}: {T_value}")

---
<a id='chapter2'></a>
# Chapter 2: Your First Simulation 🚀

Let's simulate a small system from scratch — a box of water molecules.

In [ ]:
# ============================================================
# Minimal Complete Simulation: Diatomic molecule
# ============================================================
# We'll build a 2-atom "molecule" (like H2) entirely from scratch
# to understand every component before using helper tools.

# --- Step 1: Build Topology ---
topology = app.Topology()
chain = topology.addChain()
residue = topology.addResidue('H2', chain)
atom1 = topology.addAtom('H1', app.Element.getBySymbol('H'), residue)
atom2 = topology.addAtom('H2', app.Element.getBySymbol('H'), residue)
topology.addBond(atom1, atom2)

print(f"Topology: {topology.getNumAtoms()} atoms, {topology.getNumBonds()} bonds")

# --- Step 2: Build System ---
system = openmm.System()

# Add particles with mass
mass_H = 1.008 * unit.amu
system.addParticle(mass_H)
system.addParticle(mass_H)

# Add a harmonic bond force
bond_force = openmm.HarmonicBondForce()
# addBond(atom1, atom2, equilibrium_length, spring_constant)
bond_force.addBond(0, 1, 0.074 * unit.nanometer, 500000 * unit.kilojoule_per_mole / unit.nanometer**2)
system.addForce(bond_force)

print(f"System: {system.getNumParticles()} particles, {system.getNumForces()} forces")

# --- Step 3: Choose Integrator ---
temperature = 300 * unit.kelvin
friction    = 1.0 / unit.picosecond
dt          = 0.5 * unit.femtoseconds
integrator  = openmm.LangevinMiddleIntegrator(temperature, friction, dt)

# --- Step 4: Create Context ---
platform = openmm.Platform.getPlatformByName('CPU')
context  = openmm.Context(system, integrator, platform)

# --- Step 5: Set Initial Positions ---
positions = [
    openmm.Vec3(0.0, 0.0, 0.0) * unit.nanometer,
    openmm.Vec3(0.074, 0.0, 0.0) * unit.nanometer,
]
context.setPositions(positions)

print(f"Context created on {context.getPlatform().getName()}")

# --- Step 6: Run! ---
# Get initial state
state0 = context.getState(getEnergy=True)
print(f"\nInitial potential energy: {state0.getPotentialEnergy()}")

integrator.step(1000)

state1 = context.getState(getEnergy=True, getPositions=True)
print(f"After 1000 steps:")
print(f"  Potential energy: {state1.getPotentialEnergy()}")
print(f"  Kinetic energy:   {state1.getKineticEnergy()}")
pos = state1.getPositions(asNumpy=True)
d = np.linalg.norm(pos[1] - pos[0])
print(f"  Bond length: {d*10:.4f} Å")

In [ ]:
# ============================================================
# Using the high-level Simulation wrapper
# ============================================================
# app.Simulation bundles Context + Reporters for convenience

# Re-use topology and system from above, fresh integrator
integrator2 = openmm.LangevinMiddleIntegrator(
    300 * unit.kelvin, 1.0 / unit.picosecond, 0.5 * unit.femtoseconds
)

simulation = app.Simulation(topology, system, integrator2, platform)
simulation.context.setPositions(positions)

# Add a reporter that prints to screen every 100 steps
simulation.reporters.append(
    app.StateDataReporter(
        sys.stdout, 200,
        step=True, potentialEnergy=True, temperature=True
    )
)

print("Running 1000 steps with reporter:")
simulation.step(1000)

---
<a id='chapter3'></a>
# Chapter 3: Force Fields 🔬

A force field defines the potential energy as a sum of terms:

$$U = U_{bonds} + U_{angles} + U_{torsions} + U_{electrostatics} + U_{LJ}$$

In [ ]:
# ============================================================
# Force Field Overview
# ============================================================
print("Major Force Fields Available in OpenMM:")
print()

ff_table = [
    ('amber14-all.xml',    'AMBER14',      'Proteins, nucleic acids (modern)'),
    ('amber99sb.xml',      'AMBER99SB',    'Proteins (classic, widely cited)'),
    ('amber14/protein.ff14SB.xml', 'ff14SB', 'Proteins (most accurate AMBER)'),
    ('charmm36.xml',       'CHARMM36',     'Proteins, lipids, carbohydrates'),
    ('charmm36/water.xml', 'CHARMM36 H2O', 'Water for CHARMM'),
    ('tip3p.xml',          'TIP3P',        'Water model (fast, common)'),
    ('tip4pew.xml',        'TIP4P-Ew',     'Water model (more accurate)'),
    ('tip5p.xml',          'TIP5P',        'Water model (5-site)'),
    ('spce.xml',           'SPC/E',        'Simple point charge water'),
    ('gaff.xml',           'GAFF',         'Small molecules (via OpenFF/GAFF)'),
    ('openff-2.0.0.offxml','OpenFF 2.0',   'Small molecules (via openmmforcefields)'),
]

print(f"{'File':40s} {'Name':15s} {'Use case'}")
print("-" * 80)
for f, n, u in ff_table:
    print(f"  {f:38s} {n:15s} {u}")

In [ ]:
# ============================================================
# Loading a Force Field
# ============================================================
# Common combo: protein FF + water model
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
print(f"Force field loaded: amber14-all + tip3pfb")

# CHARMM36 example
ff_charmm = app.ForceField('charmm36.xml', 'charmm36/water.xml')
print(f"Force field loaded: CHARMM36")

# Inspect the force field object
print(f"\nForce field object type: {type(forcefield)}")

In [ ]:
# ============================================================
# Force Components Explained
# ============================================================
print("Force Field Energy Terms:")
print()
print("1. BONDS (HarmonicBondForce)")
print("   U = ½ k (r - r₀)²")
print("   k = spring constant, r₀ = equilibrium bond length")
print()
print("2. ANGLES (HarmonicAngleForce)")
print("   U = ½ kθ (θ - θ₀)²")
print("   kθ = angle spring constant, θ₀ = equilibrium angle")
print()
print("3. TORSIONS (PeriodicTorsionForce)")
print("   U = Σ k [1 + cos(nφ - δ)]")
print("   n = periodicity, δ = phase")
print()
print("4. LENNARD-JONES (NonbondedForce — vdW part)")
print("   U = 4ε [(σ/r)¹² - (σ/r)⁶]")
print("   ε = well depth, σ = zero-crossing distance")
print()
print("5. ELECTROSTATICS (NonbondedForce — Coulomb part)")
print("   U = q₁q₂ / (4πε₀ r)")
print("   → Computed with PME (Particle Mesh Ewald) in periodic systems")

In [ ]:
# ============================================================
# Inspecting forces in a system
# ============================================================
# Let's create a tiny alanine dipeptide system and inspect it
import requests, tempfile

# Alanine dipeptide PDB (built-in test molecule in many MD courses)
ala_pdb_string = """ATOM      1  N   ALA     1       2.002   1.350  -0.091  1.00  0.00
ATOM      2  CA  ALA     1       1.522   0.003   0.070  1.00  0.00
ATOM      3  CB  ALA     1       2.024  -0.719   1.300  1.00  0.00
ATOM      4  C   ALA     1       0.000   0.000   0.000  1.00  0.00
ATOM      5  O   ALA     1      -0.625  -0.550   0.900  1.00  0.00
ATOM      6  N   ALA     2      -0.659   0.559  -0.984  1.00  0.00
ATOM      7  CA  ALA     2      -2.121   0.558  -1.038  1.00  0.00
ATOM      8  CB  ALA     2      -2.620  -0.872  -1.289  1.00  0.00
ATOM      9  C   ALA     2      -2.659   1.446  -2.147  1.00  0.00
ATOM     10  O   ALA     2      -1.979   2.265  -2.718  1.00  0.00
END"""

with tempfile.NamedTemporaryFile(suffix='.pdb', mode='w', delete=False) as f:
    f.write(ala_pdb_string)
    tmp_pdb = f.name

pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Create the system (vacuum, no water)
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=app.NoCutoff,
    constraints=app.HBonds,    # constrain H-bonds (allows 2 fs timestep)
    hydrogenMass=1.5*unit.amu  # HMR: allows 4 fs timestep
)

print(f"System created:")
print(f"  Particles: {system.getNumParticles()}")
print(f"  Constraints: {system.getNumConstraints()}")
print(f"  Forces:")
for i, force in enumerate(system.getForces()):
    print(f"    [{i}] {force.__class__.__name__}")

---
<a id='chapter4'></a>
# Chapter 4: System Setup from PDB 🧬

Real-world simulations start with PDB files from experiment or homology modeling.

In [ ]:
# ============================================================
# Complete PDB preparation workflow
# ============================================================
# In practice you'd use: pdb = app.PDBFile('your_protein.pdb')
# Here we simulate a minimal system

print("PDB Preparation Workflow:")
print()
print("Step 1: Get structure")
print("  - Download from RCSB PDB: https://www.rcsb.org")
print("  - Example: wget https://files.rcsb.org/download/1VII.pdb")
print()
print("Step 2: Fix the PDB file (CRITICAL!)")
print("  Common issues in PDB files:")
print("  - Missing residues/atoms")
print("  - Multiple occupancies (alt locations)")
print("  - HETATM records (ligands, waters from crystal)")
print("  - Non-standard residues")
print("  - Wrong protonation states")
print()
print("  Tools: PDBFixer (OpenMM ecosystem), pdb4amber, HTMD")

In [ ]:
# ============================================================
# Using PDBFixer (install: pip install pdbfixer)
# ============================================================
try:
    from pdbfixer import PDBFixer

    # This would be your real PDB file
    # fixer = PDBFixer(filename='1VII.pdb')
    # fixer = PDBFixer(url='https://files.rcsb.org/download/1VII.pdb')
    
    # Demonstrate with our minimal PDB
    fixer = PDBFixer(filename=tmp_pdb)
    
    # 1. Find and add missing residues
    fixer.findMissingResidues()
    print(f"Missing residues: {fixer.missingResidues}")
    
    # 2. Find non-standard residues
    fixer.findNonstandardResidues()
    print(f"Nonstandard residues: {fixer.nonstandardResidues}")
    fixer.replaceNonstandardResidues()
    
    # 3. Remove heterogens (crystallographic waters, ligands from crystal)
    fixer.removeHeterogens(keepWater=False)
    
    # 4. Find and add missing atoms
    fixer.findMissingAtoms()
    print(f"Missing atoms: {fixer.missingAtoms}")
    fixer.addMissingAtoms()
    
    # 5. Add hydrogens at target pH
    fixer.addMissingHydrogens(pH=7.0)
    
    # 6. Save fixed PDB
    with open('/tmp/fixed.pdb', 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    
    print("\nPDB fixed and saved to /tmp/fixed.pdb ✅")
    print(f"  Atoms after fixing: {fixer.topology.getNumAtoms()}")
    
except ImportError:
    print("PDBFixer not installed. Install with: pip install pdbfixer")
    print("Continuing with unfixed PDB for demonstration...")

In [ ]:
# ============================================================
# Topology Inspection
# ============================================================
pdb = app.PDBFile(tmp_pdb)

print("Topology structure:")
print(f"  Chains:   {pdb.topology.getNumChains()}")
print(f"  Residues: {pdb.topology.getNumResidues()}")
print(f"  Atoms:    {pdb.topology.getNumAtoms()}")
print(f"  Bonds:    {pdb.topology.getNumBonds()}")

print("\nChain breakdown:")
for chain in pdb.topology.chains():
    residues = list(chain.residues())
    print(f"  Chain {chain.id}: {len(residues)} residues")
    for res in residues:
        atoms = list(res.atoms())
        print(f"    Residue {res.name} {res.id}: {len(atoms)} atoms")
        for atom in atoms[:5]:
            print(f"      {atom.name:5s} (element={atom.element.symbol})")
        if len(atoms) > 5:
            print(f"      ... and {len(atoms)-5} more")

---
<a id='chapter5'></a>
# Chapter 5: Integrators & Thermostats 🌡️

Integrators advance the simulation in time. The choice affects ensemble, accuracy, and speed.

In [ ]:
# ============================================================
# Integrator Overview
# ============================================================

print("Integrators in OpenMM:")
print()

integrators = [
    ('VerletIntegrator', 'NVE', 
     'Simple leapfrog. Conserves energy. Use with thermostat.'),
    ('LangevinIntegrator', 'NVT', 
     'Stochastic thermostat. Good for equilibration.'),
    ('LangevinMiddleIntegrator', 'NVT',
     'More accurate than Langevin. RECOMMENDED for production.'),
    ('NoseHooverIntegrator', 'NVT',
     'Deterministic thermostat. Better kinetics.'),
    ('BrownianIntegrator', 'NVT (overdamped)',
     'No inertia. Useful for coarse-grained models.'),
    ('VariableLangevinIntegrator', 'NVT',
     'Adaptive timestep. Robust but slower.'),
    ('VariableVerletIntegrator', 'NVE',
     'Adaptive timestep Verlet.'),
    ('MTSIntegrator', 'multiple',
     'Multiple time stepping. Fast+slow forces on different rates.'),
]

print(f"{'Name':40s} {'Ensemble':12s} {'Notes'}")
print("-" * 90)
for name, ens, note in integrators:
    print(f"  {name:38s} {ens:12s} {note}")

In [ ]:
# ============================================================
# Creating Different Integrators
# ============================================================
T   = 300 * unit.kelvin
dt  = 2.0 * unit.femtoseconds
fric = 1.0 / unit.picosecond  # friction coefficient

# 1. Plain Verlet (NVE — no thermostat)
verlet = openmm.VerletIntegrator(dt)

# 2. Langevin (NVT)
langevin = openmm.LangevinIntegrator(T, fric, dt)
langevin.setRandomNumberSeed(42)  # for reproducibility

# 3. LangevinMiddle (NVT) — RECOMMENDED
langevin_mid = openmm.LangevinMiddleIntegrator(T, fric, dt)

# 4. Nose-Hoover (NVT)
nose_hoover = openmm.NoseHooverIntegrator(T, fric, dt)

# 5. Brownian (overdamped NVT)
brownian = openmm.BrownianIntegrator(T, fric, dt)

print("Integrators created:")
for name, integ in [('Verlet', verlet), ('Langevin', langevin), 
                     ('LangevinMiddle', langevin_mid),
                     ('NoseHoover', nose_hoover), ('Brownian', brownian)]:
    print(f"  {name:20s}: dt={integ.getStepSize()}")

In [ ]:
# ============================================================
# Choosing the Right Timestep
# ============================================================
print("Timestep Guidelines:")
print()
timestep_guide = [
    ('No constraints',                 '1 fs',  'Limited by O-H stretching (~10 fs period)'),
    ('H-bond constraints (HBonds)',    '2 fs',  'Standard choice, good balance'),
    ('All-bond constraints (AllBonds)','2-4 fs','Some accuracy compromise'),
    ('HMR + H-bond constraints',       '4 fs',  'Hydrogen Mass Repartitioning'),
    ('Coarse-grained models',          '10-20 fs', 'Depends on model'),
]
for scheme, dt_val, note in timestep_guide:
    print(f"  {scheme:40s}: {dt_val:8s} — {note}")

print()
print("Hydrogen Mass Repartitioning (HMR):")
print("  Transfer mass from heavy atoms to bonded H atoms.")
print("  Slows H-bond vibrations → allows 4 fs timestep.")
print("  Usage: hydrogenMass=3.024*unit.amu in createSystem()")

---
<a id='chapter6'></a>
# Chapter 6: Energy Minimization & Equilibration 🔋

Before production MD, we must:
1. **Minimize** — remove bad contacts from structure preparation
2. **Equilibrate** — bring temperature and pressure to target values

In [ ]:
# ============================================================
# Complete Minimization & Equilibration Protocol
# ============================================================

# Setup
pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=app.NoCutoff,  # vacuum — we'll add solvent later
    constraints=app.HBonds,
)

integrator = openmm.LangevinMiddleIntegrator(
    300 * unit.kelvin,
    1.0 / unit.picosecond,
    2.0 * unit.femtoseconds
)
integrator.setRandomNumberSeed(42)

simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

# Get initial energy
state = simulation.context.getState(getEnergy=True)
print(f"Initial potential energy: {state.getPotentialEnergy().in_units_of(unit.kilojoule_per_mole)}")

# ============================================================
# Step 1: Energy Minimization
# ============================================================
print("\nMinimizing energy...")
simulation.minimizeEnergy(
    tolerance=1.0 * unit.kilojoule_per_mole / unit.nanometer,  # convergence criterion
    maxIterations=1000
)

state = simulation.context.getState(getEnergy=True)
print(f"Post-minimization energy: {state.getPotentialEnergy().in_units_of(unit.kilojoule_per_mole)}")

In [ ]:
# ============================================================
# Step 2: Equilibration
# ============================================================
print("Equilibrating...")

# Set initial velocities from Maxwell-Boltzmann distribution at target T
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin, 42)

# Add reporter for monitoring
simulation.reporters.append(
    app.StateDataReporter(
        sys.stdout, 250,
        step=True, potentialEnergy=True, kineticEnergy=True, temperature=True,
        separator='\t'
    )
)

# Run equilibration (1000 steps = 2 ps at 2 fs/step)
simulation.step(1000)

print("\nEquilibration complete!")

# Save equilibrated structure
positions = simulation.context.getState(getPositions=True).getPositions()
app.PDBFile.writeFile(simulation.topology, positions, open('/tmp/equilibrated.pdb', 'w'))
print("Saved to /tmp/equilibrated.pdb")

In [ ]:
# ============================================================
# Temperature Annealing
# ============================================================
# Gradual heating prevents clashes and artifacts

def heat_system(simulation, T_start, T_end, n_steps, n_stages=10):
    """Gradually heat a system from T_start to T_end."""
    temps = np.linspace(T_start, T_end, n_stages + 1)
    steps_per_stage = n_steps // n_stages
    
    print(f"Heating from {T_start} K to {T_end} K in {n_stages} stages:")
    for i, T in enumerate(temps[1:], 1):
        # Set temperature in integrator
        simulation.integrator.setTemperature(T * unit.kelvin)
        simulation.step(steps_per_stage)
        
        state = simulation.context.getState(getEnergy=True)
        e = state.getPotentialEnergy().in_units_of(unit.kilojoule_per_mole)
        print(f"  Stage {i:2d}: T = {T:6.1f} K, E_pot = {e}")

# Create fresh simulation
integrator3 = openmm.LangevinMiddleIntegrator(
    0.1 * unit.kelvin, 10.0 / unit.picosecond, 1.0 * unit.femtoseconds
)
sim2 = app.Simulation(pdb.topology, system, integrator3)
sim2.context.setPositions(pdb.positions)
sim2.minimizeEnergy(maxIterations=200)

heat_system(sim2, T_start=0, T_end=300, n_steps=1000, n_stages=5)

---
<a id='chapter7'></a>
# Chapter 7: Production Runs & Reporters 📊

Reporters are the output system of OpenMM — they record what you want, when you want it.

In [ ]:
# ============================================================
# Reporter Types
# ============================================================

# Setup a simulation first
pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=app.NoCutoff, constraints=app.HBonds)
integrator = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picosecond, 2.0*unit.femtoseconds)
simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(300*unit.kelvin, 42)

# Clear any existing reporters
simulation.reporters = []

# ============================================================
# 1. StateDataReporter — energies, temperature, etc. as text
# ============================================================
simulation.reporters.append(app.StateDataReporter(
    '/tmp/state_data.csv', 100,
    step=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    temperature=True,
    volume=True,
    density=True,
    progress=True,
    remainingTime=True,
    speed=True,
    totalSteps=1000,
    separator=','
))

# Also print progress to screen
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 250,
    step=True, temperature=True, potentialEnergy=True,
    separator='\t'
))

# ============================================================
# 2. DCDReporter — binary trajectory (compact, fast)
# ============================================================
simulation.reporters.append(app.DCDReporter(
    '/tmp/trajectory.dcd', 100
))

# ============================================================
# 3. PDBReporter — human-readable trajectory (larger files)
# ============================================================
simulation.reporters.append(app.PDBReporter(
    '/tmp/trajectory.pdb', 250
))

# ============================================================
# 4. CheckpointReporter — save restart files
# ============================================================
simulation.reporters.append(app.CheckpointReporter(
    '/tmp/checkpoint.chk', 500  # save every 500 steps
))

print(f"Running production MD with {len(simulation.reporters)} reporters...")
simulation.step(1000)
print("Production run complete!")

In [ ]:
# ============================================================
# Restarting from Checkpoint
# ============================================================
print("Restarting from checkpoint...")

# Create new simulation with same setup
new_integrator = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtoseconds)
new_sim = app.Simulation(pdb.topology, system, new_integrator)

# Load checkpoint
new_sim.loadCheckpoint('/tmp/checkpoint.chk')

state = new_sim.context.getState(getEnergy=True)
print(f"Restarted at step {new_sim.currentStep}")
print(f"Energy: {state.getPotentialEnergy().in_units_of(unit.kilojoule_per_mole)}")

In [ ]:
# ============================================================
# Analyzing StateData output
# ============================================================
state_df = pd.read_csv('/tmp/state_data.csv')
state_df.columns = state_df.columns.str.strip()
print(state_df.head())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Find column names (they vary slightly)
step_col    = [c for c in state_df.columns if 'Step' in c][0]
pe_col      = [c for c in state_df.columns if 'Potential' in c][0]
temp_col    = [c for c in state_df.columns if 'Temperature' in c][0]

axes[0].plot(state_df[step_col], state_df[pe_col], 'b-', lw=1)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Potential Energy (kJ/mol)')
axes[0].set_title('Potential Energy')

axes[1].plot(state_df[step_col], state_df[temp_col], 'r-', lw=1)
axes[1].axhline(y=300, color='k', linestyle='--', alpha=0.5, label='Target 300K')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Temperature (K)')
axes[1].set_title('Temperature')
axes[1].legend()

ke_col = [c for c in state_df.columns if 'Kinetic' in c]
if ke_col:
    axes[2].plot(state_df[step_col], state_df[ke_col[0]], 'g-', lw=1)
    axes[2].set_xlabel('Step'); axes[2].set_ylabel('Kinetic Energy (kJ/mol)')
    axes[2].set_title('Kinetic Energy')

plt.tight_layout()
plt.show()

---
<a id='chapter8'></a>
# Chapter 8: Solvation & Periodic Boundary Conditions 💧

Realistic biomolecular simulations require explicit solvent and periodic boundary conditions (PBC).

In [ ]:
# ============================================================
# Adding Explicit Solvent with Modeller
# ============================================================
from openmm.app import Modeller

pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Build modeller
modeller = Modeller(pdb.topology, pdb.positions)

# Add hydrogens at pH 7
modeller.addHydrogens(forcefield, pH=7.0)
print(f"After adding H: {modeller.topology.getNumAtoms()} atoms")

# Add solvent: cubic box with 1 nm padding
modeller.addSolvent(
    forcefield,
    model='tip3p',          # water model
    padding=1.0*unit.nanometer,  # box padding around solute
    # boxSize=Vec3(5,5,5)*unit.nanometer,  # alternative: fixed box
    # boxShape='dodecahedron',  # alternative box shapes
    neutralize=True,        # add counterions to neutralize charge
    ionicStrength=0.15*unit.molar,  # physiological salt concentration
    positiveIon='Na+',
    negativeIon='Cl-'
)

print(f"After solvation: {modeller.topology.getNumAtoms()} atoms")

# Count molecule types
res_counts = {}
for res in modeller.topology.residues():
    res_counts[res.name] = res_counts.get(res.name, 0) + 1

print("\nResidue composition:")
for name, count in sorted(res_counts.items(), key=lambda x: -x[1]):
    print(f"  {name:6s}: {count}")

In [ ]:
# ============================================================
# Creating a Solvated System with PBC
# ============================================================
system_solv = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=app.PME,         # Particle Mesh Ewald for periodic systems
    nonbondedCutoff=1.0*unit.nanometer,  # real-space cutoff
    constraints=app.HBonds,
    rigidWater=True,                  # rigid water molecules (standard)
    ewaldErrorTolerance=0.0005,       # PME accuracy
)

print(f"Solvated system:")
print(f"  Particles:   {system_solv.getNumParticles()}")
print(f"  Forces:")
for force in system_solv.getForces():
    print(f"    {force.__class__.__name__}")

# Box vectors
box = modeller.topology.getPeriodicBoxVectors()
if box:
    print(f"  Box vectors:")
    for v in box:
        print(f"    {v}")

In [ ]:
# ============================================================
# Box Shapes and Minimum Image Convention
# ============================================================
print("Periodic Box Options:")
print()
print("Cube:")
print("  Most common. Easiest to visualize.")
print("  Inefficient: corners unused (30% wasted space)")
print()
print("Truncated Octahedron:")
print("  ~29% less water than cube. Faster!")
print("  boxShape='octahedron' in addSolvent()")
print()
print("Rhombic Dodecahedron:")
print("  ~71% of cube volume. Most efficient for globular proteins.")
print("  boxShape='dodecahedron' in addSolvent()")
print()
print("Key PBC rule: no atom should see itself across the box.")
print("  → Minimum box dimension > 2 × cutoff (typically > 2 nm)")
print("  → Typical padding: 1.0-1.5 nm from solute to box edge")

---
<a id='chapter9'></a>
# Chapter 9: Trajectory Analysis 📈

After simulation, we analyze trajectories to extract meaningful observables.

In [ ]:
# ============================================================
# Generate a trajectory to analyze
# ============================================================
pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system = forcefield.createSystem(pdb.topology, nonbondedMethod=app.NoCutoff, constraints=app.HBonds)
integrator = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtoseconds)
simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(300*unit.kelvin, 42)

# Record trajectory in PDB format for easy reading
simulation.reporters.append(app.DCDReporter('/tmp/traj.dcd', 50))
simulation.reporters.append(app.StateDataReporter(
    '/tmp/traj_state.csv', 50, step=True, time=True,
    potentialEnergy=True, kineticEnergy=True, temperature=True, separator=','
))

simulation.step(2000)  # 4 ps
print("Trajectory generated (40 frames)")

In [ ]:
# ============================================================
# Custom Trajectory Storage and Analysis
# ============================================================
# Sometimes you want to capture specific data during simulation

class TrajectoryCollector:
    """Custom reporter to collect positions in memory."""
    def __init__(self, interval):
        self.interval = interval
        self.positions = []
        self.times = []
        self.energies = []
    
    def describeNextReport(self, simulation):
        steps = self.interval - simulation.currentStep % self.interval
        return (steps, True, False, True, False, None)
    
    def report(self, simulation, state):
        pos = state.getPositions(asNumpy=True).value_in_unit(unit.angstrom)
        self.positions.append(pos.copy())
        self.times.append(state.getTime().value_in_unit(unit.picosecond))
        e = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        self.energies.append(e)

# Use custom reporter
integrator2 = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtoseconds)
sim2 = app.Simulation(pdb.topology, system, integrator2)
sim2.context.setPositions(pdb.positions)
sim2.minimizeEnergy()
sim2.context.setVelocitiesToTemperature(300*unit.kelvin, 42)

collector = TrajectoryCollector(100)
sim2.reporters.append(collector)
sim2.step(2000)

# Now analyze in-memory trajectory
positions_array = np.array(collector.positions)  # shape: (n_frames, n_atoms, 3)
print(f"Collected trajectory shape: {positions_array.shape}")
print(f"  n_frames = {positions_array.shape[0]}")
print(f"  n_atoms  = {positions_array.shape[1]}")
print(f"  xyz      = {positions_array.shape[2]}")

In [ ]:
# ============================================================
# RMSD Calculation
# ============================================================
# Root Mean Square Deviation from reference structure
# RMSD = sqrt(1/N Σ |r_i - r_ref_i|²)

def calc_rmsd(traj_positions, ref_positions, atom_indices=None):
    """
    Calculate RMSD of trajectory frames vs. reference.
    
    Parameters:
        traj_positions: (n_frames, n_atoms, 3)
        ref_positions:  (n_atoms, 3)
        atom_indices:   subset of atoms (e.g. CA only)
    Returns:
        RMSD array (n_frames,) in same units as input
    """
    if atom_indices is not None:
        traj_pos = traj_positions[:, atom_indices, :]
        ref_pos  = ref_positions[atom_indices, :]
    else:
        traj_pos = traj_positions
        ref_pos  = ref_positions
    
    diff = traj_pos - ref_pos[np.newaxis, :, :]
    return np.sqrt(np.mean(np.sum(diff**2, axis=2), axis=1))

# Reference = first frame
ref_pos = positions_array[0]

# All-atom RMSD
rmsd_all = calc_rmsd(positions_array, ref_pos)

# Heavy atom only (exclude H, index < threshold)
atoms_list = list(pdb.topology.atoms())
heavy_indices = [a.index for a in atoms_list if a.element.symbol != 'H']
rmsd_heavy = calc_rmsd(positions_array, ref_pos, heavy_indices)

plt.figure(figsize=(10, 4))
plt.plot(collector.times, rmsd_all,   label='All atoms',    lw=2)
plt.plot(collector.times, rmsd_heavy, label='Heavy atoms',  lw=2, linestyle='--')
plt.xlabel('Time (ps)')
plt.ylabel('RMSD (Å)')
plt.title('RMSD vs. First Frame')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean RMSD (heavy atoms): {rmsd_heavy.mean():.3f} Å")
print(f"Max  RMSD (heavy atoms): {rmsd_heavy.max():.3f} Å")

In [ ]:
# ============================================================
# RMSF — Per-Atom Fluctuations
# ============================================================
# RMSF measures how much each atom moves around its average position
# High RMSF → flexible region; Low RMSF → rigid region

def calc_rmsf(traj_positions):
    """Calculate per-atom RMSF from trajectory."""
    mean_pos = traj_positions.mean(axis=0)  # (n_atoms, 3)
    diff = traj_positions - mean_pos[np.newaxis, :, :]
    return np.sqrt(np.mean(np.sum(diff**2, axis=2), axis=0))  # (n_atoms,)

rmsf = calc_rmsf(positions_array)

atoms_names = [a.name for a in pdb.topology.atoms()]
res_names   = [a.residue.name + str(a.residue.id) for a in pdb.topology.atoms()]

plt.figure(figsize=(12, 4))
plt.bar(range(len(rmsf)), rmsf, color='steelblue', alpha=0.8)
plt.xlabel('Atom Index')
plt.ylabel('RMSF (Å)')
plt.title('Per-Atom RMSF')
plt.xticks(range(len(atoms_names)), atoms_names, rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

print("Most flexible atoms:")
top5_idx = np.argsort(rmsf)[::-1][:5]
for idx in top5_idx:
    print(f"  Atom {idx:3d} ({atoms_names[idx]:4s} in {res_names[idx]}): RMSF = {rmsf[idx]:.3f} Å")

In [ ]:
# ============================================================
# Distance, Angle, and Dihedral Tracking
# ============================================================
def calc_distance(pos1, pos2):
    """Distance between two atom positions."""
    return np.linalg.norm(pos2 - pos1)

def calc_angle_deg(pos1, pos2, pos3):
    """Angle between three atoms (in degrees)."""
    v1 = pos1 - pos2
    v3 = pos3 - pos2
    cos_angle = np.dot(v1, v3) / (np.linalg.norm(v1) * np.linalg.norm(v3))
    return np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

def calc_dihedral_deg(pos1, pos2, pos3, pos4):
    """Dihedral angle between four atoms (in degrees)."""
    b1 = pos2 - pos1
    b2 = pos3 - pos2
    b3 = pos4 - pos3
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    n1 /= np.linalg.norm(n1)
    n2 /= np.linalg.norm(n2)
    cos_d = np.dot(n1, n2)
    sign = np.sign(np.dot(b1, n2))
    return sign * np.degrees(np.arccos(np.clip(cos_d, -1, 1)))

# Track distances/angles over time
distances = []
angles    = []

for frame_pos in positions_array:
    # Distance between first N and last O
    n_atoms = frame_pos.shape[0]
    dist = calc_distance(frame_pos[0], frame_pos[n_atoms-1])
    distances.append(dist)
    
    # Angle for atoms 0, 1, 2
    if n_atoms >= 3:
        ang = calc_angle_deg(frame_pos[0], frame_pos[1], frame_pos[2])
        angles.append(ang)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(collector.times, distances, 'b-', lw=1.5)
ax1.set_xlabel('Time (ps)'); ax1.set_ylabel('Distance (Å)')
ax1.set_title('N-terminus to C-terminus Distance')
ax1.grid(True, alpha=0.3)

if angles:
    ax2.plot(collector.times, angles, 'r-', lw=1.5)
    ax2.set_xlabel('Time (ps)'); ax2.set_ylabel('Angle (°)')
    ax2.set_title('N-CA-CB Angle')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Using MDTraj for Advanced Analysis (recommended tool)
# ============================================================
try:
    import mdtraj as md
    
    # Load trajectory
    traj = md.load('/tmp/traj.dcd', top=tmp_pdb)
    print(f"MDTraj trajectory:")
    print(f"  n_frames:    {traj.n_frames}")
    print(f"  n_atoms:     {traj.n_atoms}")
    print(f"  n_residues:  {traj.n_residues}")
    print(f"  time:        {traj.time[0]:.2f} to {traj.time[-1]:.2f} ps")
    
    # RMSD with MDTraj
    rmsd_mdtraj = md.rmsd(traj, traj, 0)  # vs frame 0
    print(f"\nMDTraj RMSD: {rmsd_mdtraj.mean()*10:.3f} Å (mean)")
    
    # RMSF
    rmsf_mdtraj = md.rmsf(traj, traj, 0)
    print(f"MDTraj RMSF: {rmsf_mdtraj.mean()*10:.3f} Å (mean)")
    
    # Radii of gyration
    rg = md.compute_rg(traj)
    print(f"Rg: {rg.mean()*10:.3f} Å (mean)")
    
    # Secondary structure (DSSP)
    dssp = md.compute_dssp(traj[0])
    print(f"\nDSSP secondary structure (frame 0):")
    for res, ss in zip(traj.topology.residues, dssp[0]):
        ss_name = {'H': 'Alpha-helix', 'E': 'Beta-sheet', 'C': 'Coil'}.get(ss, ss)
        print(f"  {res.name}{res.resSeq}: {ss_name}")

except ImportError:
    print("MDTraj not installed. Install with: conda install -c conda-forge mdtraj")
    print("MDTraj provides efficient trajectory analysis including RMSD, RMSF, DSSP, contacts, etc.")

---
<a id='chapter10'></a>
# Chapter 10: Constraints, Restraints & Custom Forces 🔗

Restraints guide or restrict molecular motion during simulation.

In [ ]:
# ============================================================
# Constraints vs Restraints
# ============================================================
print("Constraints vs Restraints:")
print()
print("CONSTRAINTS (system.addConstraint):")
print("  - HARD: bond/angle held exactly at fixed value")
print("  - Implemented via SHAKE/LINCS algorithms")
print("  - Used for H-bonds to allow longer timesteps")
print("  - Zero computational overhead vs explicit bonds")
print()
print("RESTRAINTS (as additional Force):")
print("  - SOFT: harmonic penalty for deviation from target")
print("  - U = k(x - x0)² — molecule can still move, penalized")
print("  - Common uses:")
print("    · Position restraints: prevent large movements during equilibration")
print("    · Distance restraints: enforce known contacts (NMR data)")
print("    · Dihedral restraints: maintain conformation")

In [ ]:
# ============================================================
# Position Restraints
# ============================================================
# Commonly used to: equilibrate solvent while holding protein fixed

pdb = app.PDBFile(tmp_pdb)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

system = forcefield.createSystem(
    pdb.topology, nonbondedMethod=app.NoCutoff, constraints=app.HBonds
)

# Add position restraints to heavy atoms
restraint_force = openmm.CustomExternalForce(
    'k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)'
)
restraint_force.addGlobalParameter('k', 1000.0 * unit.kilojoule_per_mole / unit.nanometer**2)
restraint_force.addPerParticleParameter('x0')
restraint_force.addPerParticleParameter('y0')
restraint_force.addPerParticleParameter('z0')

# Apply to all heavy atoms
pos = pdb.positions
for atom in pdb.topology.atoms():
    if atom.element.symbol != 'H':
        x0 = pos[atom.index].x
        y0 = pos[atom.index].y
        z0 = pos[atom.index].z
        restraint_force.addParticle(atom.index, [x0, y0, z0])

system.addForce(restraint_force)
print(f"Added position restraints to {restraint_force.getNumParticles()} heavy atoms")

# To remove restraints: change k to 0 or remove the force
# simulation.context.setParameter('k', 0.0)  # turn off softly
# Or remove the force entirely before production

In [ ]:
# ============================================================
# Custom Forces — the most powerful feature of OpenMM
# ============================================================
# CustomForce expressions are compiled to machine code!

# Example 1: Custom nonbonded force (soft-core LJ)
soft_lj = openmm.CustomNonbondedForce(
    '4*epsilon*((sigma/r)^12 - (sigma/r)^6); '
    'sigma=0.5*(sigma1+sigma2); epsilon=sqrt(epsilon1*epsilon2)'
)
soft_lj.addPerParticleParameter('sigma')
soft_lj.addPerParticleParameter('epsilon')
print("Custom LJ force created")

# Example 2: Distance restraint between two atoms
dist_restraint = openmm.CustomBondForce('k*(r-r0)^2')
dist_restraint.addGlobalParameter('k', 1000 * unit.kilojoule_per_mole / unit.nanometer**2)
dist_restraint.addPerBondParameter('r0')
dist_restraint.addBond(0, 5, [0.5])  # atoms 0 and 5, target distance 0.5 nm
print("Distance restraint force created")

# Example 3: Flat-bottom restraint (no penalty within range)
flat_bottom = openmm.CustomBondForce(
    'k * max(0, r-r_max)^2 + k * max(0, r_min-r)^2'
)
flat_bottom.addGlobalParameter('k', 100 * unit.kilojoule_per_mole / unit.nanometer**2)
flat_bottom.addPerBondParameter('r_min')
flat_bottom.addPerBondParameter('r_max')
flat_bottom.addBond(0, 1, [0.3, 0.7])  # allowed range: 0.3-0.7 nm
print("Flat-bottom restraint created")

# Example 4: Custom torsion (Ramachandran-like)
custom_torsion = openmm.CustomTorsionForce('k*(1 + cos(n*theta - theta0))')
custom_torsion.addGlobalParameter('k', 5 * unit.kilojoule_per_mole)
custom_torsion.addGlobalParameter('n', 3)
custom_torsion.addGlobalParameter('theta0', 0.0)
print("Custom torsion force created")

In [ ]:
# ============================================================
# Umbrella Sampling Setup
# ============================================================
# Umbrella sampling: apply harmonic bias along a reaction coordinate
# to sample rare events. Commonly used for PMF calculations.

def create_umbrella_simulation(system, topology, positions, 
                                atom_i, atom_j, r_center, k_umbrella):
    """
    Add umbrella potential along distance between two atoms.
    Returns (simulation, umbrella_force)
    """
    import copy
    sys_copy = copy.deepcopy(system)
    
    # Harmonic bias: k*(r - r0)^2
    umbrella = openmm.CustomBondForce('k_umb*(r-r0_umb)^2')
    umbrella.addGlobalParameter('k_umb', k_umbrella)
    umbrella.addGlobalParameter('r0_umb', r_center)
    umbrella.addBond(atom_i, atom_j)
    sys_copy.addForce(umbrella)
    
    integ = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtoseconds)
    sim = app.Simulation(topology, sys_copy, integ)
    sim.context.setPositions(positions)
    return sim, umbrella

# Example: umbrella windows along N-C terminal distance
centers = np.arange(3.0, 8.0, 0.5)  # 3 to 8 Å in 0.5 Å steps
k_umb   = 1000 * unit.kilojoule_per_mole / unit.nanometer**2

print("Umbrella Sampling Windows:")
print(f"  Reaction coordinate: atom 0 to atom {pdb.topology.getNumAtoms()-1}")
print(f"  Windows: {len(centers)}")
print(f"  Centers: {centers} Å")
print(f"  Force constant: {k_umb}")

# In practice, you'd run each window independently
# then use WHAM or MBAR to reconstruct the PMF

---
<a id='chapter11'></a>
# Chapter 11: Barostats & NPT Ensemble 🌡️

Real biological systems at constant pressure require a barostat (pressure controller).

In [ ]:
# ============================================================
# NPT Simulation (constant N, P, T)
# ============================================================
# Most production runs use NPT:
# - Allows box to fluctuate → correct liquid density
# - Required for membrane/lipid simulations
# - Standard for biomolecular free energy calculations

# Build a solvated system (using our earlier modeller)
modeller_npt = Modeller(pdb.topology, pdb.positions)
modeller_npt.addHydrogens(forcefield)
modeller_npt.addSolvent(forcefield, model='tip3p', padding=0.5*unit.nanometer,
                         neutralize=True)

ff_npt = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system_npt = ff_npt.createSystem(
    modeller_npt.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=app.HBonds,
    rigidWater=True,
)

# Add Monte Carlo Barostat for constant pressure
barostat = openmm.MonteCarloBarostat(
    1.0 * unit.bar,          # target pressure
    300 * unit.kelvin,        # temperature (must match integrator)
    25                        # frequency: attempt volume change every 25 steps
)
system_npt.addForce(barostat)
print(f"Added barostat: {barostat.__class__.__name__}")

# Create NPT simulation
integrator_npt = openmm.LangevinMiddleIntegrator(
    300*unit.kelvin, 1.0/unit.picosecond, 2.0*unit.femtoseconds
)
sim_npt = app.Simulation(modeller_npt.topology, system_npt, integrator_npt)
sim_npt.context.setPositions(modeller_npt.positions)

print(f"System: {system_npt.getNumParticles()} particles")

# Verify barostat is present
print("Forces:")
for f in system_npt.getForces():
    print(f"  {f.__class__.__name__}")

In [ ]:
# ============================================================
# Anisotropic Barostat (for membranes)
# ============================================================
# Membranes require different pressure in Z vs. XY
aniso_barostat = openmm.MonteCarloAnisotropicBarostat(
    openmm.Vec3(1, 1, 1) * unit.bar,  # x, y, z pressures
    300 * unit.kelvin,
    scaleX=True, scaleY=True, scaleZ=True  # which axes to scale
)
print(f"Anisotropic barostat: {aniso_barostat.__class__.__name__}")

# For membranes, typically:
# aniso_barostat = MonteCarloAnisotropicBarostat(
#     Vec3(1,1,1)*bar, 303*kelvin,
#     scaleX=True, scaleY=True, scaleZ=False  # don't scale Z independently
# )

# Membrane barostat
# membrane_barostat = MonteCarloMembraneBarostat(
#     1*bar,                               # normal pressure
#     0*bar*nanometer,                     # surface tension (0 for bilayer)
#     300*kelvin,
#     MonteCarloMembraneBarostat.XYIsotropic,  # XY coupling
#     MonteCarloMembraneBarostat.ZFree        # Z independent
# )
print("\nFor membrane simulations:")
print("  Use MonteCarloMembraneBarostat with surface tension = 0")

---
<a id='chapter12'></a>
# Chapter 12: Enhanced Sampling Methods 🎲

Standard MD can get stuck in local minima. Enhanced sampling overcomes energy barriers.

In [ ]:
# ============================================================
# Overview of Enhanced Sampling Methods
# ============================================================
print("Enhanced Sampling Methods:")
print()
methods = [
    ('Replica Exchange MD (REMD)',
     'Run parallel replicas at different T, swap configurations.',
     'Protein folding, conformational sampling'),
    ('Metadynamics',
     'Add history-dependent bias to escape local minima.',
     'Free energy surfaces, dihedral sampling'),
    ('Well-Tempered Metadynamics',
     'Improved metadynamics with decreasing hill heights.',
     'More converged free energy estimates'),
    ('Umbrella Sampling',
     'Apply harmonic bias at intervals along reaction coord.',
     'PMF calculations, binding/unbinding'),
    ('Steered MD (SMD)',
     'Apply constant force to pull a molecule along a direction.',
     'Unbinding, pore passage, mechanical unfolding'),
    ('Accelerated MD (aMD)',
     'Boost dihedral potential to accelerate conformational changes.',
     'Large-scale conformational changes'),
    ('Simulated Tempering',
     'Dynamically change temperature during single run.',
     'Alternative to REMD'),
]
for name, desc, app_str in methods:
    print(f"  {name}")
    print(f"    → {desc}")
    print(f"    → Use for: {app_str}")
    print()

In [ ]:
# ============================================================
# Metadynamics (basic implementation)
# ============================================================
# Metadynamics adds Gaussians to the free energy surface
# to discourage revisiting explored regions

class SimpleMetadynamics:
    """
    Minimal 1D metadynamics implementation.
    Bias collective variable: distance between atom i and j.
    """
    def __init__(self, simulation, atom_i, atom_j,
                 sigma=0.05, height=0.5, deposit_freq=100):
        self.sim = simulation
        self.atom_i = atom_i
        self.atom_j = atom_j
        self.sigma = sigma        # Gaussian width (nm)
        self.height = height      # Gaussian height (kJ/mol)
        self.deposit_freq = deposit_freq
        
        # Bias force using tabulated values
        self.bias_force = openmm.CustomBondForce('bias_e')
        self.bias_force.addBond(atom_i, atom_j)
        self.gaussians = []  # (center, height, sigma)
        
        # Track CV values
        self.cv_history = []
        self.fes_history = []
    
    def get_cv(self):
        """Get current value of collective variable (distance)."""
        state = self.sim.context.getState(getPositions=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
        return np.linalg.norm(pos[self.atom_j] - pos[self.atom_i])
    
    def calc_bias(self, r):
        """Calculate current bias energy at position r."""
        bias = 0.0
        for center, h, s in self.gaussians:
            bias += h * np.exp(-(r - center)**2 / (2 * s**2))
        return bias
    
    def run(self, n_steps):
        """Run metadynamics simulation."""
        for step in range(n_steps):
            self.sim.step(1)
            cv = self.get_cv()
            self.cv_history.append(cv)
            
            # Deposit Gaussian at current position
            if step % self.deposit_freq == 0:
                self.gaussians.append((cv, self.height, self.sigma))
                
        return self.cv_history

# Setup and run
pdb2 = app.PDBFile(tmp_pdb)
system2 = forcefield.createSystem(pdb2.topology, nonbondedMethod=app.NoCutoff, constraints=app.HBonds)
integ2 = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 10/unit.picosecond, 2*unit.femtoseconds)
sim_meta = app.Simulation(pdb2.topology, system2, integ2)
sim_meta.context.setPositions(pdb2.positions)
sim_meta.minimizeEnergy()

meta = SimpleMetadynamics(sim_meta, atom_i=0, atom_j=pdb2.topology.getNumAtoms()-1,
                            sigma=0.02, height=0.1, deposit_freq=50)
cv_traj = meta.run(500)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(cv_traj, 'b-', lw=1, alpha=0.7)
plt.xlabel('Step'); plt.ylabel('CV (nm)')
plt.title('CV Evolution (Metadynamics)')

plt.subplot(1, 2, 2)
# Approximate free energy surface (negative bias is FES)
r_vals = np.linspace(min(cv_traj)*0.9, max(cv_traj)*1.1, 100)
fes = [-meta.calc_bias(r) for r in r_vals]
fes = np.array(fes) - min(fes)
plt.plot(r_vals * 10, fes, 'r-', lw=2)
plt.xlabel('Distance (Å)'); plt.ylabel('Free Energy (kJ/mol)')
plt.title('Approximate FES')

plt.tight_layout()
plt.show()
print(f"Deposited {len(meta.gaussians)} Gaussians")

In [ ]:
# ============================================================
# Steered MD (Constant Velocity Pulling)
# ============================================================
# Apply a moving harmonic restraint to pull atoms apart

class SteeredMD:
    """
    Constant-velocity steered MD along atom-atom distance.
    Records force over displacement → work from Jarzynski equality.
    """
    def __init__(self, simulation, atom_i, atom_j, 
                 pull_speed, k_smd, steps_per_sample=10):
        self.sim = simulation
        self.atom_i = atom_i
        self.atom_j = atom_j
        self.pull_speed = pull_speed   # nm/step
        self.k_smd = k_smd             # kJ/mol/nm²
        self.steps_per_sample = steps_per_sample
        
        # Create moving restraint
        self.smd_force = openmm.CustomBondForce(
            'k_smd * (r - r0_smd)^2'
        )
        self.smd_force.addGlobalParameter('k_smd', k_smd)
        self.smd_force.addGlobalParameter('r0_smd', 0.0)
        self.smd_force.setUsesPeriodicBoundaryConditions(False)
        self.smd_force.addBond(atom_i, atom_j)
        
        idx = simulation.system.addForce(self.smd_force)
        simulation.context.reinitialize(preserveState=True)
        
        self.time = []
        self.cv_values = []
        self.force_values = []
    
    def _get_distance(self):
        state = self.sim.context.getState(getPositions=True)
        pos = state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
        return np.linalg.norm(pos[self.atom_j] - pos[self.atom_i])
    
    def run(self, n_steps):
        r0 = self._get_distance()
        self.sim.context.setParameter('r0_smd', r0)
        
        for s in range(n_steps):
            r0 += self.pull_speed
            self.sim.context.setParameter('r0_smd', r0)
            self.sim.step(self.steps_per_sample)
            
            r = self._get_distance()
            f = 2 * self.k_smd * (r - r0)  # dU/dr
            
            self.time.append(s)
            self.cv_values.append(r)
            self.force_values.append(-f)  # negative = pulling force

print("Steered MD class ready.")
print("Usage:")
print("  smd = SteeredMD(sim, atom_i=0, atom_j=9,")
print("                  pull_speed=0.001, k_smd=1000)")
print("  smd.run(100)")
print("  # Work = integral of force × displacement")
print("  # ΔG = -kT ln <exp(-W/kT)>  (Jarzynski equality)")

---
<a id='chapter13'></a>
# Chapter 13: Free Energy Calculations ⚡

Free energy differences are the gold standard for predicting binding affinities, solvation energies, and thermodynamic properties.

In [ ]:
# ============================================================
# Free Energy Perturbation (FEP) / Alchemical Methods
# ============================================================
# Alchemical: gradually "mutate" one molecule into another
# λ=0: molecule A; λ=1: molecule B
# Run simulations at multiple λ values, compute ΔG

print("Alchemical Free Energy Methods:")
print()
print("Thermodynamic Cycle:")
print("  Ligand(aq)  + Protein → Protein:Ligand(aq)")
print("       ↕ ΔG_solv          ↕ ΔG_bind")
print("  Ligand(vac) + Protein → Protein:Ligand(vac)")
print()
print("  ΔΔG_bind = ΔG_protein - ΔG_water")
print()
print("Lambda schedule (typical 12 windows):")
lambdas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print(f"  λ = {lambdas}")

In [ ]:
# ============================================================
# Alchemical System with Soft-Core Potentials
# ============================================================
# OpenMM supports alchemical transformations via custom forces

def create_alchemical_system(system, alchemical_atoms):
    """
    Create an alchemical system where specified atoms are decoupled.
    Uses a simple linear scaling approach (for demonstration).
    In practice, use openmmtools.alchemy for proper soft-core.
    """
    import copy
    alch_system = copy.deepcopy(system)
    
    # Add lambda parameter
    # Modify NonbondedForce to scale interactions
    for i, force in enumerate(alch_system.getForces()):
        if isinstance(force, openmm.NonbondedForce):
            # For each alchemical atom, scale charge and epsilon
            # This is simplified — real alchemical needs soft-core VDW
            for atom_idx in alchemical_atoms:
                charge, sigma, epsilon = force.getParticleParameters(atom_idx)
                # Store original parameters as custom parameters
                force.setParticleParameters(atom_idx, charge, sigma, epsilon)
    
    return alch_system

print("Alchemical system creation demonstrated.")
print()
print("For production FEP calculations, use:")
print("  - openmmtools: pip install openmmtools")
print("    → AlchemicalFactory, AbsoluteAlchemicalFactory")
print("  - perses: conda install -c conda-forge perses")
print("    → Full relative FEP pipeline")
print("  - OpenFE: pip install openfe")
print("    → Modern FEP framework")

In [ ]:
# ============================================================
# Bennett Acceptance Ratio (BAR) — FEP analysis
# ============================================================
# BAR gives minimum variance ΔG estimate from adjacent λ windows

def bar_estimate(dU_forward, dU_reverse, beta, max_iter=1000, tol=1e-6):
    """
    Bennett Acceptance Ratio for ΔG estimation.
    
    dU_forward: energy differences from state A evaluated in state B
    dU_reverse: energy differences from state B evaluated in state A
    beta: 1/(kT)
    """
    n_f = len(dU_forward)
    n_r = len(dU_reverse)
    
    M = np.log(n_f / n_r)
    C = 0.0  # initial guess
    
    for _ in range(max_iter):
        f_fwd = 1.0 / (1.0 + np.exp(beta * (dU_forward - C - M)))
        f_rev = 1.0 / (1.0 + np.exp(-beta * (dU_reverse + C - M)))
        C_new = (1/beta) * (np.log(f_rev.mean()) - np.log(f_fwd.mean())) + C
        if abs(C_new - C) < tol:
            break
        C = C_new
    
    return C_new  # ΔG = C + kT * ln(n_f/n_r)

# Demonstrate with synthetic data
kT = 2.479  # kJ/mol at 298 K
beta = 1.0 / kT
true_dG = 5.0  # kJ/mol

np.random.seed(42)
dU_fwd = np.random.normal(true_dG, 2.0, 500)
dU_rev = np.random.normal(-true_dG, 2.0, 500)

dG_bar = bar_estimate(dU_fwd, dU_rev, beta)
dG_fep = -np.log(np.exp(-beta * dU_fwd).mean()) / beta  # FEP estimate

print(f"True ΔG:  {true_dG:.3f} kJ/mol")
print(f"FEP ΔG:   {dG_fep:.3f} kJ/mol")
print(f"BAR ΔG:   {dG_bar:.3f} kJ/mol")
print(f"\nBAR is more accurate than simple FEP (lower variance).")

In [ ]:
# ============================================================
# MBAR — Multistate Bennett Acceptance Ratio
# ============================================================
try:
    from pymbar import MBAR, timeseries
    
    print("pymbar available for MBAR analysis")
    print("MBAR is the gold standard for multi-state FE calculations")
    print()
    print("Usage with OpenMM data:")
    print("  1. Run simulations at λ = 0, 0.1, 0.2, ..., 1.0")
    print("  2. For each frame, compute energy at ALL λ values")
    print("  3. Feed u_kln matrix to MBAR:")
    print("     mbar = MBAR(u_kln, N_k)")
    print("     results = mbar.compute_free_energy_differences()")
    
except ImportError:
    print("pymbar not installed: pip install pymbar")
    print("pymbar implements MBAR, BAR, and TI analysis")

---
<a id='chapter14'></a>
# Chapter 14: GPU Acceleration & Performance ⚡

In [ ]:
# ============================================================
# Platform Selection and Benchmarking
# ============================================================

def benchmark_platform(platform_name, n_steps=1000, properties=None):
    """Benchmark simulation speed on a given platform."""
    try:
        platform = openmm.Platform.getPlatformByName(platform_name)
        
        pdb_b = app.PDBFile(tmp_pdb)
        ff_b  = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
        sys_b = ff_b.createSystem(pdb_b.topology, nonbondedMethod=app.NoCutoff, constraints=app.HBonds)
        intg_b = openmm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtoseconds)
        
        if properties:
            sim_b = app.Simulation(pdb_b.topology, sys_b, intg_b, platform, properties)
        else:
            sim_b = app.Simulation(pdb_b.topology, sys_b, intg_b, platform)
        
        sim_b.context.setPositions(pdb_b.positions)
        sim_b.minimizeEnergy(maxIterations=100)
        
        t0 = time.time()
        sim_b.step(n_steps)
        elapsed = time.time() - t0
        
        ns_per_day = (n_steps * 2e-6 / elapsed) * 86400
        print(f"  {platform_name:12s}: {elapsed:.3f}s ({ns_per_day:.1f} ns/day)")
        return elapsed
    except Exception as e:
        print(f"  {platform_name:12s}: Not available ({e})")
        return None

print("Platform Benchmarks:")
benchmark_platform('CPU')
benchmark_platform('CUDA')      # NVIDIA GPU
benchmark_platform('OpenCL')    # AMD/Intel GPU
benchmark_platform('Reference') # Slow reference implementation (for validation)

In [ ]:
# ============================================================
# GPU Platform Properties
# ============================================================
print("GPU Platform Properties (CUDA):")
try:
    cuda = openmm.Platform.getPlatformByName('CUDA')
    for prop in cuda.getPropertyNames():
        print(f"  {prop}")
    
    # Optimized GPU properties
    cuda_props = {
        'DeviceIndex': '0',         # GPU index (0 = first GPU)
        'Precision': 'mixed',       # float32 for forces, float64 for positions
                                    # 'single' = fastest, 'double' = most accurate
        'UseCpuPme': 'false',       # PME on GPU
        'DeterministicForces': 'false',  # True for exact reproducibility
    }
    print("\nRecommended GPU properties:")
    for k, v in cuda_props.items():
        print(f"  {k}: {v}")
except:
    print("  CUDA not available on this machine")

print("\nTo use GPU:")
print("  platform = Platform.getPlatformByName('CUDA')")
print("  properties = {'DeviceIndex': '0', 'Precision': 'mixed'}")
print("  simulation = Simulation(topology, system, integrator, platform, properties)")

In [ ]:
# ============================================================
# Performance Tips
# ============================================================
print("Performance Optimization Checklist:")
print()
tips = [
    ('Timestep',        'Use 2 fs (HBonds) or 4 fs (HMR) — never 1 fs in production'),
    ('Constraints',     'Use HBonds or AllBonds + rigidWater=True'),
    ('HMR',             'hydrogenMass=3.024*amu → 4 fs timestep (3× speedup)'),
    ('Cutoff',          '0.9-1.2 nm; shorter = faster, longer = more accurate'),
    ('PME tolerance',   'ewaldErrorTolerance=0.0005 (default); 0.001 for speed'),
    ('Box shape',       'Truncated octahedron → ~29% fewer waters → faster'),
    ('GPU precision',   'mixed precision: 2-3× faster than double'),
    ('Minimize first',  'Always minimize before dynamics to avoid crashes'),
    ('Equilibrate',     '5-10 ns equilibration before production'),
    ('Checkpoint',      'Checkpoint every 1-2 ns to enable recovery from crashes'),
    ('DCD reporters',   'Write DCD every 1-10 ps; too frequent = disk bottleneck'),
    ('CPU threads',     'setNumThreads in CPU platform props for multi-core'),
]
for item, tip in tips:
    print(f"  ✓ {item:20s}: {tip}")

---
<a id='chapter15'></a>
# Chapter 15: Advanced Topics & Real Workflows 🏆

In [ ]:
# ============================================================
# 15.1 Custom Integrator
# ============================================================
# OpenMM lets you write your OWN integrator in a simple language!

# Velocity Verlet integrator (same as built-in VerletIntegrator but explicit)
dt = 2.0 * unit.femtoseconds

custom_verlet = openmm.CustomIntegrator(dt)
custom_verlet.addUpdateContextState()  # update constraints, barostat etc.
custom_verlet.addComputePerDof("v", "v+0.5*dt*f/m")   # v half-step
custom_verlet.addComputePerDof("x", "x+dt*v")           # update positions
custom_verlet.addComputePerDof("v", "v+0.5*dt*f/m")   # v full-step
custom_verlet.addConstrainPositions()
custom_verlet.addConstrainVelocities()

print("Custom Velocity Verlet integrator created")

# BAOAB Langevin integrator (alternative to built-in)
gamma = 1.0  # friction in ps^-1
kT = 2.479   # kJ/mol at 300 K

baoab = openmm.CustomIntegrator(dt)
baoab.addGlobalVariable("kT", kT)
baoab.addGlobalVariable("gamma", gamma)
baoab.addPerDofVariable("sigma", 0)
baoab.addComputePerDof("sigma", "sqrt(kT/m)")
baoab.addUpdateContextState()
# B: half-kick
baoab.addComputePerDof("v", "v + 0.5*dt*f/m")
# A: half-drift
baoab.addComputePerDof("x", "x + 0.5*dt*v")
# O: full Ornstein-Uhlenbeck
baoab.addComputePerDof("v", "v*exp(-gamma*dt) + sigma*sqrt(1-exp(-2*gamma*dt))*gaussian")
# A: half-drift
baoab.addComputePerDof("x", "x + 0.5*dt*v")
# B: half-kick
baoab.addComputePerDof("v", "v + 0.5*dt*f/m")
baoab.addConstrainPositions()
baoab.addConstrainVelocities()

print("BAOAB Langevin integrator created")

In [ ]:
# ============================================================
# 15.2 Complete Production Workflow Template
# ============================================================
def run_production_simulation(
    pdb_file,
    ff_files=['amber14-all.xml', 'amber14/tip3pfb.xml'],
    output_prefix='/tmp/production',
    temperature=300,
    pressure=1.0,
    n_equil_steps=5000,
    n_prod_steps=10000,
    report_freq=500,
    dcd_freq=100,
    dt_fs=2.0,
    platform_name='CPU'
):
    """
    Complete OpenMM simulation protocol:
    1. Load and prepare structure
    2. Add solvent
    3. Create solvated system with PME
    4. Energy minimize
    5. NVT equilibration (protein restrained)
    6. NPT equilibration (unrestrained)
    7. NPT production run
    """
    print(f"{'='*60}")
    print(f"OpenMM Production Workflow")
    print(f"{'='*60}")
    print(f"  PDB: {pdb_file}")
    print(f"  FF:  {', '.join(ff_files)}")
    print(f"  T:   {temperature} K")
    print(f"  P:   {pressure} bar")
    print(f"  dt:  {dt_fs} fs")
    print()
    
    # 1. Load structure
    pdb = app.PDBFile(pdb_file)
    ff  = app.ForceField(*ff_files)
    
    # 2. Prepare with Modeller
    modeller = Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(ff, pH=7.0)
    print(f"Step 1: Added H atoms → {modeller.topology.getNumAtoms()} atoms")
    
    modeller.addSolvent(ff, model='tip3p', padding=1.0*unit.nanometer,
                        neutralize=True, ionicStrength=0.15*unit.molar)
    print(f"Step 2: Solvated → {modeller.topology.getNumAtoms()} atoms")
    
    # 3. Create system
    system = ff.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometer,
        constraints=app.HBonds,
        rigidWater=True,
        ewaldErrorTolerance=0.0005,
    )
    
    # 4. Add barostat for NPT
    system.addForce(openmm.MonteCarloBarostat(
        pressure*unit.bar, temperature*unit.kelvin, 25
    ))
    print(f"Step 3: System created with {system.getNumParticles()} particles")
    
    # 5. Setup simulation
    integrator = openmm.LangevinMiddleIntegrator(
        temperature*unit.kelvin, 1.0/unit.picosecond, dt_fs*unit.femtoseconds
    )
    platform = openmm.Platform.getPlatformByName(platform_name)
    simulation = app.Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(modeller.positions)
    
    # 6. Minimize
    print("Step 4: Minimizing...")
    simulation.minimizeEnergy(tolerance=10*unit.kilojoule_per_mole/unit.nanometer)
    
    # 7. Set velocities
    simulation.context.setVelocitiesToTemperature(temperature*unit.kelvin)
    
    # 8. Equilibrate
    print(f"Step 5: Equilibrating ({n_equil_steps} steps)...")
    equil_reporter = app.StateDataReporter(
        f"{output_prefix}_equil.csv", report_freq,
        step=True, time=True, temperature=True,
        potentialEnergy=True, density=True, separator=','
    )
    simulation.reporters.append(equil_reporter)
    simulation.step(n_equil_steps)
    simulation.reporters.clear()
    print(f"  Equilibration complete")
    
    # 9. Production
    print(f"Step 6: Production run ({n_prod_steps} steps)...")
    simulation.reporters.append(app.DCDReporter(f"{output_prefix}.dcd", dcd_freq))
    simulation.reporters.append(app.StateDataReporter(
        f"{output_prefix}_prod.csv", report_freq,
        step=True, time=True, temperature=True,
        potentialEnergy=True, kineticEnergy=True, density=True,
        speed=True, progress=True, totalSteps=n_prod_steps, separator=','
    ))
    simulation.reporters.append(app.CheckpointReporter(
        f"{output_prefix}.chk", report_freq*10
    ))
    
    t0 = time.time()
    simulation.step(n_prod_steps)
    elapsed = time.time() - t0
    
    # 10. Save final state
    positions = simulation.context.getState(getPositions=True).getPositions()
    app.PDBFile.writeFile(simulation.topology, positions,
                          open(f"{output_prefix}_final.pdb", 'w'))
    
    sim_time_ns = n_prod_steps * dt_fs * 1e-6
    ns_per_day = sim_time_ns / elapsed * 86400
    print(f"\nProduction complete:")
    print(f"  Simulated: {sim_time_ns:.4f} ns")
    print(f"  Wall time: {elapsed:.1f}s")
    print(f"  Speed:     {ns_per_day:.2f} ns/day")
    print(f"  Output:    {output_prefix}.dcd")
    
    return simulation

# Run the workflow!
sim_final = run_production_simulation(
    pdb_file=tmp_pdb,
    output_prefix='/tmp/prod_run',
    n_equil_steps=500,
    n_prod_steps=1000,
    report_freq=200,
    dcd_freq=50,
)

In [ ]:
# ============================================================
# 15.3 Small Molecule Parameterization
# ============================================================
print("Small Molecule Force Fields (ligands):")
print()
print("Option 1: GAFF2 via openmmforcefields")
print("  from openmmforcefields.generators import GAFFTemplateGenerator")
print("  from openff.toolkit.topology import Molecule")
print("  mol = Molecule.from_smiles('CCO')")
print("  gaff = GAFFTemplateGenerator(molecules=[mol], forcefield='gaff-2.11')")
print("  ff = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')")
print("  ff.registerTemplateGenerator(gaff.generator)")
print()
print("Option 2: OpenFF (SMIRNOFF) — modern approach")
print("  from openmmforcefields.generators import SMIRNOFFTemplateGenerator")
print("  from openff.toolkit.topology import Molecule")
print("  mol = Molecule.from_smiles('CCO')")
print("  smirnoff = SMIRNOFFTemplateGenerator(molecules=[mol],")
print("              forcefield='openff-2.0.0')")
print()
print("Option 3: CGenFF (CHARMM)")
print("  Use ParamChem web server: https://cgenff.umaryland.edu")
print("  Then load with: ff = ForceField('charmm36.xml', 'ligand.xml')")

In [ ]:
# ============================================================
# 15.4 Protein-Ligand Binding Simulation
# ============================================================
print("Protein-Ligand Simulation Protocol:")
print()
print("1. Dock ligand into binding site (AutoDock, Glide, Vina)")
print("2. Prepare complex PDB (protein + docked ligand)")
print("3. Parameterize ligand (GAFF2 or OpenFF)")
print("4. Run with combined force field:")
print()

code_example = '''
from openmmforcefields.generators import GAFFTemplateGenerator
from openff.toolkit.topology import Molecule

# Load complex
complex_pdb = app.PDBFile('protein_ligand.pdb')

# Parameterize ligand
ligand = Molecule.from_smiles('CC(=O)Oc1ccccc1C(=O)O')  # your ligand
gaff = GAFFTemplateGenerator(molecules=[ligand])

# Combined force field
ff = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
ff.registerTemplateGenerator(gaff.generator)

# Solvate and run normally...
modeller = Modeller(complex_pdb.topology, complex_pdb.positions)
modeller.addSolvent(ff, padding=1.0*unit.nanometer)
system = ff.createSystem(modeller.topology, nonbondedMethod=app.PME, ...)
'''
print(code_example)

In [ ]:
# ============================================================
# 15.5 OpenMM Ecosystem Map
# ============================================================
print("The OpenMM Ecosystem:")
print()

ecosystem = [
    ('openmmforcefields',  'conda-forge', 'GAFF2, SMIRNOFF, CHARMM for small molecules'),
    ('openff-toolkit',     'conda-forge', 'Open Force Field — modern parameterization'),
    ('pdbfixer',           'conda-forge', 'PDB preparation, missing atoms/residues'),
    ('mdtraj',             'conda-forge', 'Trajectory analysis: RMSD, RMSF, DSSP, etc.'),
    ('parmed',             'conda-forge', 'Force field parameter inspection and editing'),
    ('pymbar',             'pip/conda',   'BAR/MBAR free energy analysis'),
    ('openmmtools',        'conda-forge', 'Alchemical FEP, REMD, NCMC'),
    ('perses',             'conda-forge', 'Automated relative FEP calculations'),
    ('openfe',             'pip',         'Modern FEP framework (RBFE)'),
    ('htmd',               'pip',         'High-throughput MD setup and analysis'),
    ('nglview',            'conda-forge', 'Jupyter-based 3D molecular visualization'),
    ('openmmdl',           'pip',         'MD setup pipeline for proteins and ligands'),
    ('plumed',             'conda-forge', 'Metadynamics, CV analysis (via interface)'),
]

print(f"{'Package':25s} {'Install':15s} {'Purpose'}")
print("-" * 80)
for pkg, install, purpose in ecosystem:
    print(f"  {pkg:23s} {install:15s} {purpose}")

---

# 🎓 Summary & Key Takeaways

You've completed the OpenMM zero-to-hero journey! Here's what you've mastered:

| Topic | Key Classes / Functions |
|-------|------------------------|
| Architecture | `Topology`, `System`, `Integrator`, `Context`, `Simulation` |
| Units | `unit.nanometer`, `unit.kelvin`, `unit.kilojoule_per_mole` |
| Force Fields | `app.ForceField()`, `forcefield.createSystem()` |
| Structure prep | `app.PDBFile()`, `Modeller`, `PDBFixer` |
| Integrators | `LangevinMiddleIntegrator`, `VerletIntegrator`, `NoseHooverIntegrator` |
| Minimization | `simulation.minimizeEnergy()` |
| Equilibration | `setVelocitiesToTemperature()`, annealing |
| Reporters | `StateDataReporter`, `DCDReporter`, `CheckpointReporter` |
| Solvation | `Modeller.addSolvent()`, PBC, PME |
| Analysis | RMSD, RMSF, distances, MDTraj |
| Restraints | `CustomExternalForce`, `CustomBondForce` |
| Barostat | `MonteCarloBarostat`, NPT ensemble |
| Enhanced sampling | Metadynamics, Steered MD, Umbrella sampling |
| Free energy | FEP, BAR, MBAR, alchemical methods |
| GPU | CUDA platform, mixed precision |
| Custom forces | `CustomNonbondedForce`, `CustomIntegrator` |

## Typical Workflow Summary
```
PDB file → PDBFixer → Modeller.addHydrogens() → Modeller.addSolvent()
  → ForceField.createSystem() + MonteCarloBarostat
  → LangevinMiddleIntegrator → Simulation
  → minimizeEnergy() → NVT equil (restrained) → NPT equil
  → Production (DCD + StateData + Checkpoint)
  → MDTraj analysis (RMSD, RMSF, contacts, secondary structure)
```

## Key Resources
- 📖 [OpenMM Documentation](http://docs.openmm.org)
- 🏃 [OpenMM Cookbook](https://openmm.github.io/openmm-cookbook/)
- 💬 [OpenMM GitHub Discussions](https://github.com/openmm/openmm/discussions)
- 📦 [OpenFF Toolkit Docs](https://docs.openforcefield.org)
- 📊 [MDTraj Docs](https://mdtraj.org)
- 📐 [Alchemistry wiki](https://alchemistry.org)

---
*Happy simulating! ⚗️🧬*